<a href="https://colab.research.google.com/github/Fabi-22/si3003-workshop-rag-Fabi-22/blob/main/notebooks/02b_rag_app_nvidia_nim_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Aplicación RAG con NVIDIA NIM en Google Colab

Esta versión es **autosuficiente**: configura NVIDIA NIM, carga los documentos, crea un índice Qdrant temporal y publica una interfaz Gradio.

**Modelos:**
- Embeddings: `nvidia/nemotron-3-embed-1b`
- Generación: `nvidia/nemotron-3-super-120b-a12b`

> En Colab, ejecuta las celdas en orden. La última celda mostrará una URL pública temporal de Gradio.


## 1. Instalar dependencias

Después de la instalación no suele ser necesario reiniciar el runtime. Si Colab lo solicita, reinícialo y vuelve a ejecutar desde la celda siguiente.


In [1]:
%pip install -qU gradio langchain langchain-core langchain-text-splitters langchain-nvidia-ai-endpoints langchain-qdrant qdrant-client python-dotenv


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.3/89.3 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 406.5/406.5 kB 9.0 MB/s eta 0:00:00


## 2. Configurar la API key

En Colab, abre **🔑 Secrets**, crea un secreto llamado `NVIDIA_API_KEY` y habilita el acceso para este notebook. También puedes introducirla de forma oculta cuando la celda la solicite.


In [30]:
import os
import json
import getpass
from pathlib import Path
from typing import List, Dict

import gradio as gr
from dotenv import load_dotenv
from langchain_nvidia_ai_endpoints import NVIDIAEmbeddings, ChatNVIDIA
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableParallel
from langchain_text_splitters import RecursiveCharacterTextSplitter
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams
from langchain_qdrant import QdrantVectorStore

load_dotenv()

try:
    from google.colab import userdata
    NVIDIA_API_KEY = userdata.get("NVIDIA_API_KEY").strip()
except (ImportError, KeyError):
    NVIDIA_API_KEY = os.getenv("NVIDIA_API_KEY")

if not NVIDIA_API_KEY:
    NVIDIA_API_KEY = getpass.getpass("NVIDIA_API_KEY: ").strip()
if not NVIDIA_API_KEY:
    raise ValueError("Falta NVIDIA_API_KEY. Créala en https://build.nvidia.com/")

os.environ["NVIDIA_API_KEY"] = NVIDIA_API_KEY
GENERATION_MODEL = "nvidia/nemotron-3.5-super-vl-120b-a12b"
EMBEDDING_MODEL = "nvidia/llama-nemotron-embed-vl-1b-v2"
COLLECTION = "company_docs_gradio"

print("✅ Configuración NVIDIA NIM lista")


✅ Configuración NVIDIA NIM lista


## 3. Cargar los documentos

Si quieres usar documentos propios, súbelos a `data/company_documents.json` con los campos `id`, `title`, `content` y, opcionalmente, `metadata`. Si el archivo no existe, la aplicación usa un conjunto de demostración.


In [6]:
DEMO_DOCUMENTS = [
    {
        "id": "product_001", "title": "CloudSync Pro — Precios",
        "content": "El plan Enterprise cuesta $49/mes por usuario con compromiso anual. El plan mensual sin compromiso cuesta $59/mes por usuario.",
        "metadata": {"department": "sales", "type": "pricing"},
    },
    {
        "id": "hr_001", "title": "Política de trabajo remoto",
        "content": "Los empleados pueden trabajar de forma remota hasta 3 días por semana, con aprobación del jefe directo.",
        "metadata": {"department": "hr", "type": "policy"},
    },
    {
        "id": "finance_001", "title": "Política de reembolsos",
        "content": "Los reembolsos mayores a $100 requieren aprobación del supervisor. Una vez aprobados, se procesan en 3 a 5 días hábiles al método de pago original.",
        "metadata": {"department": "finance", "type": "policy"},
    },
    {
        "id": "hr_002", "title": "Onboarding",
        "content": "Durante la primera semana se completan módulos obligatorios de seguridad, cumplimiento y cultura de la empresa.",
        "metadata": {"department": "hr", "type": "procedure"},
    },
]


def load_documents(path: Path) -> List[Dict]:
    if not path.exists():
        print(f"ℹ️ No se encontró {path}; se usarán documentos de demostración.")
        return DEMO_DOCUMENTS

    with path.open(encoding="utf-8") as file:
        documents = json.load(file)

    required = {"id", "title", "content"}
    for index, document in enumerate(documents):
        missing = required - document.keys()
        if missing:
            raise ValueError(f"Al documento {index} le faltan campos: {missing}")
        document.setdefault("metadata", {})
    return documents


company_documents = load_documents(Path("company_documents.json"))
print(f"✅ {len(company_documents)} documentos disponibles")


ℹ️ No se encontró company_documents.json; se usarán documentos de demostración.
✅ 4 documentos disponibles


## 4. Construir el pipeline RAG

Esta celda crea los fragmentos, calcula embeddings con NVIDIA NIM y los indexa en Qdrant. El índice vive en memoria mientras la sesión de Colab esté activa.


In [31]:
company_documents = DEMO_DOCUMENTS  # Baseline A1; A2 carga el corpus propio en su celda.
def build_pipeline(documents: List[Dict]):
    embeddings = NVIDIAEmbeddings(model=EMBEDDING_MODEL, truncate="END")
    llm = ChatNVIDIA(
        model=GENERATION_MODEL,
        temperature=0.2,
        top_p=0.95,
        max_completion_tokens=700,
    )

    llm = llm.bind(chat_template_kwargs={"enable_thinking": False})
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=400,
        chunk_overlap=40,
        separators=["\n\n", "\n", ".", " "],
    )

    chunks: List[Document] = []
    for document in documents:
        for index, text in enumerate(splitter.split_text(document["content"])):
            chunks.append(Document(
                page_content=text,
                metadata={
                    "source_id": document["id"],
                    "title": document["title"],
                    "chunk_index": index,
                    **document.get("metadata", {}),
                },
            ))

    # La dimensión se obtiene del modelo y no queda codificada manualmente.
    embedding_dim = len(embeddings.embed_query("prueba de dimensión"))
    qdrant_client = QdrantClient(":memory:")
    qdrant_client.create_collection(
        collection_name=COLLECTION,
        vectors_config=VectorParams(size=embedding_dim, distance=Distance.COSINE),
    )

    vectorstore = QdrantVectorStore(
        client=qdrant_client,
        collection_name=COLLECTION,
        embedding=embeddings,
    )
    vectorstore.add_documents(chunks)

    retriever = vectorstore.as_retriever(
        search_type="similarity_score_threshold",
        search_kwargs={"k": 3, "score_threshold": 0.3},
    )

    prompt = ChatPromptTemplate.from_messages([
        (
            "system",
            "Eres un asistente empresarial. Responde únicamente con información "
            "del contexto proporcionado. Si el contexto no contiene la respuesta, "
            "indica que no tienes esa información. Responde en español y de forma concisa.",
        ),
        ("human", "CONTEXTO:\n{context}\n\nPREGUNTA:\n{question}"),
    ])

    def format_docs(items: List[Document]) -> str:
        return "\n\n---\n\n".join(
            f"[{item.metadata.get('title', 'Sin título')}]\n{item.page_content}"
            for item in items
        )

    setup_and_retrieval = RunnableParallel(
        question=RunnablePassthrough(),
        documents=retriever,
    )
    answer_from_documents = (
        {
            "context": lambda item: format_docs(item["documents"]),
            "question": lambda item: item["question"],
        }
        | prompt
        | llm
        | StrOutputParser()
    )
    pipeline = setup_and_retrieval.assign(answer=answer_from_documents)
    return pipeline, chunks


print("Construyendo el índice...")
rag_pipeline, indexed_chunks = build_pipeline(company_documents)
titles = [document["title"] for document in company_documents]
print(f"✅ Pipeline listo: {len(titles)} documentos, {len(indexed_chunks)} chunks")


Construyendo el índice...


/usr/local/lib/python3.13/dist-packages/langchain_nvidia_ai_endpoints/_common.py:269: UserWarning: Found nvidia/llama-nemotron-embed-vl-1b-v2 in available_models, but type is unknown and inference may fail.
  warnings.warn(


✅ Pipeline listo: 4 documentos, 4 chunks


## 5. Crear la interfaz Gradio

In [26]:
def chat(question: str, history: list | None):
    history = history or []
    question = (question or "").strip()
    if not question:
        return "", history, "Escribe una pregunta."

    try:
        result = rag_pipeline.invoke(question)
        answer = result["answer"].strip()
        documents = result["documents"]

        unique_sources = []
        for document in documents:
            title = document.metadata.get("title", "Sin título")
            if title not in unique_sources:
                unique_sources.append(title)

        sources_md = "**Fuentes recuperadas:**\n" + "\n".join(
            f"- {title}" for title in unique_sources
        ) if unique_sources else "No se encontraron documentos relevantes."

    except Exception as error:
        answer = "Ocurrió un error al consultar el pipeline. Revisa el detalle mostrado debajo."
        sources_md = f"**Error:** `{type(error).__name__}: {error}`"

    history = history + [
        {"role": "user", "content": question},
        {"role": "assistant", "content": answer},
    ]
    return "", history, sources_md


def clear_chat():
    return [], ""


with gr.Blocks(title="RAG empresarial con NVIDIA NIM") as demo:
    gr.Markdown(
        "# RAG empresarial\n"
        "Consulta documentos internos mediante **LangChain + NVIDIA NIM + Qdrant**."
    )

    with gr.Row():
        with gr.Column(scale=1):
            gr.Markdown("### Documentos indexados")
            gr.Markdown("\n".join(f"- {title}" for title in titles))
            gr.Markdown(f"*{len(indexed_chunks)} chunks en el índice*")
            gr.Markdown("---")
            gr.Markdown(
                f"**Embeddings:** `{EMBEDDING_MODEL}`  \n"
                f"**LLM:** `{GENERATION_MODEL}`  \n"
                "**Vector store:** Qdrant en memoria  \n"
                "**Retrieval:** k=3, threshold=0.3"
            )

        with gr.Column(scale=2):
            # Gradio 6 usa mensajes {role, content} por defecto;
            # el antiguo argumento type="messages" ya no existe.
            chatbot = gr.Chatbot(height=430, label="Chat")
            with gr.Row():
                question_box = gr.Textbox(
                    placeholder="Haz una pregunta sobre los documentos...",
                    label="Pregunta",
                    lines=2,
                    scale=4,
                )
                send_button = gr.Button("Enviar", variant="primary", scale=1)

            sources_box = gr.Markdown()
            clear_button = gr.Button("Limpiar", variant="secondary")

    gr.Examples(
        examples=[
            ["¿Cuál es el precio del plan Enterprise de CloudSync Pro?"],
            ["¿Cuántos días pueden trabajar remotamente los empleados?"],
            ["¿Qué pasa con los reembolsos mayores a $100?"],
            ["¿Qué ocurre durante la primera semana de onboarding?"],
        ],
        inputs=question_box,
        label="Preguntas de ejemplo",
    )

    send_button.click(
        chat,
        inputs=[question_box, chatbot],
        outputs=[question_box, chatbot, sources_box],
    )
    question_box.submit(
        chat,
        inputs=[question_box, chatbot],
        outputs=[question_box, chatbot, sources_box],
    )
    clear_button.click(clear_chat, outputs=[chatbot, sources_box])


## 6. Ejecutar la aplicación en Colab

`share=True` crea el enlace público temporal. La aplicación estará disponible mientras esta celda y el runtime de Colab continúen activos.


In [27]:
demo.queue().launch(
    share=True,
    debug=True,
    show_error=True,
)


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://52ecf76dd597657bf5.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://52ecf76dd597657bf5.gradio.live


### Detener o volver a ejecutar

- Interrumpe la última celda para detener el servidor.
- Antes de volver a ejecutarla, puedes cerrar instancias previas con `gr.close_all()`.
- El enlace público cambia cada vez que se inicia la aplicación.


In [ ]:
# Ejecuta esta celda solo cuando quieras cerrar servidores Gradio anteriores.
gr.close_all()


In [23]:
EMBEDDING_MODEL = "nvidia/llama-nemotron-embed-vl-1b-v2"
GENERATION_MODEL = "nvidia/nemotron-3.5-super-vl-120b-a12b"
# A2–A4: experimentos reproducibles; se conserva el pipeline demo de A1 arriba.
from time import sleep
from datetime import datetime, timezone
A_RESULTS = {'model': GENERATION_MODEL, 'embedding_model': EMBEDDING_MODEL}
# El corpus se incorpora abajo como JSON; no contiene secretos.
CORPUS = [{'id': 'planes', 'title': 'Planes de Café Nube', 'content': 'Café Nube es una empresa ficticia de suscripción de café colombiano. El plan Origen cuesta 45000 pesos colombianos al mes e incluye una bolsa de 340 gramos. El plan Explorador cuesta 82000 pesos mensuales e incluye dos bolsas de 340 gramos de regiones distintas. Los precios incluyen IVA. No existe plan empresarial.', 'metadata': {'department': 'ventas', 'type': 'precios'}}, {'id': 'envios', 'title': 'Cobertura y tiempos de envío', 'content': 'Café Nube realiza envíos a Medellín, Bogotá y Cali. El envío cuesta 8000 pesos por pedido y es gratuito desde 80000 pesos en productos. Los pedidos se despachan dentro de dos días hábiles después del pago. La entrega toma de dos a cuatro días hábiles adicionales. No se realizan envíos internacionales.', 'metadata': {'department': 'logistica', 'type': 'procedimiento'}}, {'id': 'devoluciones', 'title': 'Cambios y devoluciones', 'content': 'Si una bolsa llega abierta o el pedido es incorrecto, el cliente debe informar a Café Nube dentro de cinco días calendario de la entrega y adjuntar una fotografía del producto y el número del pedido. Se ofrece reposición sin costo o devolución del dinero. El reintegro se procesa en siete días hábiles después de aprobar el caso.', 'metadata': {'department': 'servicio', 'type': 'politica'}}, {'id': 'pausas', 'title': 'Pausa y cancelación de suscripciones', 'content': 'La suscripción de Café Nube se puede pausar durante un máximo de dos meses consecutivos. Para que no se genere el siguiente cobro, la pausa o cancelación debe solicitarse con al menos 72 horas de anticipación a la fecha de renovación. No existe penalidad por cancelar. Un pedido ya despachado no se cancela por este procedimiento.', 'metadata': {'department': 'servicio', 'type': 'politica'}}, {'id': 'talleres', 'title': 'Taller de métodos de preparación', 'content': 'Café Nube ofrece un taller presencial de métodos de preparación los sábados de 10:00 a 12:00 en su sede ficticia de Medellín. Cada sesión admite ocho participantes y cuesta 60000 pesos por persona. La inscripción se paga por adelantado. Para reprogramar sin costo se requiere avisar al menos 24 horas antes.', 'metadata': {'department': 'formacion', 'type': 'actividad'}}, {'id': 'soporte', 'title': 'Horario y canales de soporte', 'content': 'El soporte de Café Nube atiende de lunes a viernes de 8:00 a 17:00, excepto festivos de Colombia, mediante el formulario de ayuda de la tienda. El tiempo objetivo de primera respuesta es de un día hábil. Una solicitud de soporte debe incluir el número del pedido; nunca se solicita la contraseña ni datos completos de la tarjeta.', 'metadata': {'department': 'servicio', 'type': 'procedimiento'}}, {'id': 'almacenamiento', 'title': 'Conservación y molienda del café', 'content': 'Café Nube recomienda guardar el café en un recipiente hermético, en un lugar fresco y seco, alejado de la luz. No recomienda refrigerarlo. Cada bolsa indica fecha de tueste y lote. El café se ofrece en grano, molienda para filtro o molienda para prensa francesa; la elección debe hacerse antes de pagar el pedido.', 'metadata': {'department': 'producto', 'type': 'recomendacion'}}, {'id': 'fidelidad', 'title': 'Programa de fidelidad Semillas', 'content': 'Por cada 10000 pesos pagados en productos de Café Nube se obtiene una Semilla; el costo del envío no acumula puntos. Diez Semillas se pueden canjear por un descuento de 15000 pesos en productos. Las Semillas vencen seis meses después de su emisión. El descuento no se combina con otras promociones.', 'metadata': {'department': 'ventas', 'type': 'programa'}}]
Path('data').mkdir(exist_ok=True)
Path('data/company_documents.json').write_text(json.dumps(CORPUS, ensure_ascii=False, indent=2), encoding='utf-8')
company_documents = load_documents(Path('data/company_documents.json'))
PROMPT_BEFORE = ('Eres un asistente empresarial. Responde únicamente con información del contexto proporcionado. Si el contexto no contiene la respuesta, indica que no tienes esa información. Responde en español y de forma concisa.')
PROMPT_AFTER = ('Eres un asistente empresarial. Usa únicamente el contexto proporcionado. Cita el título exacto de cada documento que respalda tu respuesta entre corchetes. Si el contexto no alcanza para responder, escribe exactamente: No tengo información suficiente en los documentos para responder. No completes datos con conocimiento externo. Responde en español con un máximo de 3 oraciones.')

def build_pipeline(documents, system_prompt=PROMPT_BEFORE):
    embeddings = NVIDIAEmbeddings(model=EMBEDDING_MODEL, truncate='END')
    llm = ChatNVIDIA(model=GENERATION_MODEL, temperature=0.2, top_p=0.95, max_completion_tokens=700)
    splitter = RecursiveCharacterTextSplitter(chunk_size=400, chunk_overlap=40, separators=['\n\n','\n','.',' '])
    chunks=[]
    for document in documents:
        for index,text in enumerate(splitter.split_text(document['content'])):
            chunks.append(Document(page_content=text,metadata={'source_id':document['id'],'title':document['title'],'chunk_index':index,**document.get('metadata',{})}))
    embedding_dim=len(embeddings.embed_query('prueba de dimensión'))
    client=QdrantClient(':memory:')
    client.create_collection(collection_name=COLLECTION,vectors_config=VectorParams(size=embedding_dim,distance=Distance.COSINE))
    vectorstore=QdrantVectorStore(client=client,collection_name=COLLECTION,embedding=embeddings)
    vectorstore.add_documents(chunks)
    return chain_for_store(vectorstore, system_prompt), chunks, vectorstore

def chain_for_store(store, system_prompt, threshold=0.3):
    retriever=store.as_retriever(search_type='similarity_score_threshold',search_kwargs={'k':3,'score_threshold':threshold})
    prompt=ChatPromptTemplate.from_messages([('system',system_prompt),('human','CONTEXTO:\n{context}\n\nPREGUNTA:\n{question}')])
    def formatted(items):
        return '\n\n---\n\n'.join(f"[{d.metadata.get('title','Sin título')}]\n{d.page_content}" for d in items)
    setup=RunnableParallel(question=RunnablePassthrough(),documents=retriever)
    answer=({'context':lambda item:formatted(item['documents']),'question':lambda item:item['question']}|prompt|ChatNVIDIA(model=GENERATION_MODEL,temperature=0.2,max_completion_tokens=700).bind(chat_template_kwargs={'enable_thinking':False})|StrOutputParser())
    return setup.assign(answer=answer)

rag_pipeline,indexed_chunks,vectorstore=build_pipeline(company_documents)
titles=[d['title'] for d in company_documents]
A_RESULTS['corpus_chunks']=len(indexed_chunks)
QUESTIONS_A2=[
'¿Cuánto cuesta el plan Explorador y qué incluye?',
'¿Cuánto vale el envío de un pedido de 90000 pesos en productos a Bogotá?',
'¿Cuándo debo pedir una pausa para evitar el próximo cobro?',
'¿Cuántas personas admite el taller y cuánto cuesta por persona?',
'¿Cuántas Semillas necesito para un descuento y de cuánto es?',
'¿Cuál es el NIT legal de Café Nube?',
'¿Qué tasa de interés cobra Café Nube por financiar una compra?']

def ask_record(chain,q):
    for attempt in range(4):
        try:
            r=chain.invoke(q)
            break
        except Exception as error:
            if attempt==3: raise
            print('Reintento NVIDIA',attempt+1,type(error).__name__)
            sleep(2**attempt)
    return {'question':q,'answer':r['answer'],'sources':[d.metadata for d in r['documents']],'contexts':[d.page_content for d in r['documents']]}

A_RESULTS['A2']=[]
for q in QUESTIONS_A2:
    row=ask_record(rag_pipeline,q); A_RESULTS['A2'].append(row)
    print('A2',json.dumps(row,ensure_ascii=False));sleep(0.5)

QUESTIONS_A3=[QUESTIONS_A2[0],'¿Cuánto cuesta el servicio?',QUESTIONS_A2[5]]
A_RESULTS['A3_before']=[]
for q in QUESTIONS_A3:
    row=ask_record(rag_pipeline,q);A_RESULTS['A3_before'].append(row);print('A3 antes',json.dumps(row,ensure_ascii=False));sleep(0.5)
# Solo cambia el prompt: se reutilizan vectores y retriever.
rag_pipeline=chain_for_store(vectorstore,PROMPT_AFTER)
A_RESULTS['A3_after']=[]
for q in QUESTIONS_A3:
    row=ask_record(rag_pipeline,q);A_RESULTS['A3_after'].append(row);print('A3 después',json.dumps(row,ensure_ascii=False));sleep(0.5)
A_RESULTS['prompt_before']=PROMPT_BEFORE;A_RESULTS['prompt_after']=PROMPT_AFTER
A_RESULTS['A4']=[]
for q in [QUESTIONS_A2[0],QUESTIONS_A2[2],QUESTIONS_A2[3],QUESTIONS_A2[5],QUESTIONS_A2[6]]:
    scored=vectorstore.similarity_search_with_score(q,k=3)
    row={'question':q,'scores':[{'title':d.metadata['title'],'score':float(s),'chunk_index':d.metadata['chunk_index']} for d,s in scored]}
    A_RESULTS['A4'].append(row);print('A4',json.dumps(row,ensure_ascii=False));sleep(0.5)
Path('resultados_A.json').write_text(json.dumps(A_RESULTS,ensure_ascii=False,indent=2),encoding='utf-8')
print('✅ A2, A3 y A4 ejecutados. Resultados guardados en resultados_A.json')

print("EXPORT_A=" + json.dumps(A_RESULTS,ensure_ascii=True))


/usr/local/lib/python3.13/dist-packages/langchain_nvidia_ai_endpoints/_common.py:269: UserWarning: Found nvidia/llama-nemotron-embed-vl-1b-v2 in available_models, but type is unknown and inference may fail.
  warnings.warn(


A2 {"question": "¿Cuánto cuesta el plan Explorador y qué incluye?", "answer": "El plan Explorador cuesta 82.000 pesos colombianos mensuales e incluye dos bolsas de 340 gramos de regiones distintas. Los precios incluyen IVA.", "sources": [{"source_id": "planes", "title": "Planes de Café Nube", "chunk_index": 0, "department": "ventas", "type": "precios", "_id": "0e21555a8e574d3ea58d61201d284f5a", "_collection_name": "company_docs_gradio"}, {"source_id": "talleres", "title": "Taller de métodos de preparación", "chunk_index": 0, "department": "formacion", "type": "actividad", "_id": "87abc38f824b49428b3c465ba4a814c7", "_collection_name": "company_docs_gradio"}, {"source_id": "fidelidad", "title": "Programa de fidelidad Semillas", "chunk_index": 0, "department": "ventas", "type": "programa", "_id": "3931a0708c7f4969b92de90b03901f0f", "_collection_name": "company_docs_gradio"}], "contexts": ["Café Nube es una empresa ficticia de suscripción de café colombiano. El plan Origen cuesta 45000 pes

In [25]:
# A1 repetido con los dos modelos definitivos solicitados.
corpus_pipeline,corpus_chunks,corpus_store=rag_pipeline,indexed_chunks,vectorstore
corpus_titles=titles
rag_pipeline,indexed_chunks,vectorstore=build_pipeline(DEMO_DOCUMENTS,PROMPT_BEFORE)
titles=[d['title'] for d in DEMO_DOCUMENTS]
A1_FINAL=[]
for question in ['¿Cuál es el precio del plan Enterprise de CloudSync Pro?','¿Cuántos días pueden trabajar remotamente los empleados?','¿Qué pasa con los reembolsos mayores a $100?','¿Qué ocurre durante la primera semana de onboarding?','¿Cuál es el NIT legal de CloudSync Pro?']:
    row=ask_record(rag_pipeline,question)
    A1_FINAL.append(row)
    print('A1 final',json.dumps(row,ensure_ascii=False),flush=True)
    sleep(0.5)
A_RESULTS['A1']=A1_FINAL
A_RESULTS['A1_chunks']=len(indexed_chunks)
print('EXPORT_A='+json.dumps(A_RESULTS,ensure_ascii=True),flush=True)
print('✅ A1 repetido con configuración final',flush=True)


A1 final {"question": "¿Cuál es el precio del plan Enterprise de CloudSync Pro?", "answer": "El plan Enterprise cuesta $49/mes por usuario con compromiso anual.", "sources": [{"source_id": "product_001", "title": "CloudSync Pro — Precios", "chunk_index": 0, "department": "sales", "type": "pricing", "_id": "615c46b7e2c641bd8509a9583eedd3f9", "_collection_name": "company_docs_gradio"}, {"source_id": "finance_001", "title": "Política de reembolsos", "chunk_index": 0, "department": "finance", "type": "policy", "_id": "d6f8c547debf4b388dcec4732031dca4", "_collection_name": "company_docs_gradio"}, {"source_id": "hr_001", "title": "Política de trabajo remoto", "chunk_index": 0, "department": "hr", "type": "policy", "_id": "b55360a31af6425f9e8b98fa00269366", "_collection_name": "company_docs_gradio"}], "contexts": ["El plan Enterprise cuesta $49/mes por usuario con compromiso anual. El plan mensual sin compromiso cuesta $59/mes por usuario.", "Los reembolsos mayores a $100 requieren aprobación

In [29]:
# A2/A4: restaurar el índice del corpus sin llamar otra vez embeddings.
rag_pipeline,indexed_chunks,vectorstore=corpus_pipeline,corpus_chunks,corpus_store
titles=corpus_titles

def chat(question,history):
    history=history or []
    question=(question or '').strip()
    if not question:return '',history,'Escribe una pregunta.'
    result=ask_record(rag_pipeline,question)
    scored=vectorstore.similarity_search_with_score(question,k=3)
    sources_md='**Fuentes recuperadas y scores coseno:**\n'+ '\n'.join(f"- **{doc.metadata['title']}** · chunk {doc.metadata['chunk_index']} · score **{score:.4f}**" for doc,score in scored)
    history=history+[{'role':'user','content':question},{'role':'assistant','content':result['answer']}]
    return '',history,sources_md
with gr.Blocks(title='Café Nube · RAG con scores · Fabiola Valencia Barrios') as demo_corpus:
    gr.Markdown('# Café Nube · RAG empresarial\nCorpus ficticio de Fabiola Valencia Barrios · SI3003')
    with gr.Row():
        with gr.Column(scale=1):
            gr.Markdown('### Documentos indexados\n'+'\n'.join('- '+t for t in titles))
            gr.Markdown(f'{len(indexed_chunks)} chunks · chunk_size=400 · overlap=40\n\n**Embeddings:** `{EMBEDDING_MODEL}`\n\n**LLM:** `{GENERATION_MODEL}`\n\n**Recuperación:** k=3, umbral=0.30 (referencia de los experimentos). Umbral propuesto: 0.32, pendiente de validar con más preguntas.')
        with gr.Column(scale=2):
            chatbot=gr.Chatbot(height=360,label='Conversación')
            question_box=gr.Textbox(label='Pregunta',value='¿Cuánto cuesta el plan Explorador y qué incluye?')
            send=gr.Button('Enviar',variant='primary')
            sources_box=gr.Markdown()
            send.click(chat,[question_box,chatbot],[question_box,chatbot,sources_box])
            question_box.submit(chat,[question_box,chatbot],[question_box,chatbot,sources_box])
demo_corpus.queue().launch(share=True,debug=False,show_error=True)


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://2979810b543aefffe9.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
